In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from nnsight import LanguageModel
import numpy as np
from transformers.generation.utils import GenerationConfig
import json
import torch
from tqdm import tqdm
from transformers.tokenization_utils import AddedToken, PreTrainedTokenizer

/home/wxy/.conda/envs/wxy_llm_sample/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model_name = 'llama2-7b-chat-hf'
# model_name = 'Mistral-7B-Instruct-v0.2'
model_path = f'/mnt/local/xywang/models/{model_name}'


base_model = AutoModelForCausalLM.from_pretrained(model_path, low_cpu_mem_usage=True, torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(model_path)
# if tokenizer.pad_token is None:
#     tokenizer.add_special_tokens({'pad_token': '[PAD]'})
#     base_model.resize_token_embeddings(len(tokenizer))

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model.to('cuda')
# base_model.eval()
model = LanguageModel(base_model, tokenizer=tokenizer)

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  7.80it/s]


In [ ]:
model = LanguageModel(model_path)

In [6]:
question = 'Mason is trying to download a 880 MB game to his phone. After downloading 310 MB, his Internet connection slows to 3 MB/minute. How many more minutes will it take him to download the game?'
index = 0
all_hidden_states = []
all_attention_states = []

prompt = tokenizer([question], return_tensors = 'pt').input_ids

with model.trace(prompt):
    for layer in model.model.layers:
        all_attention_states.append(layer.self_attn.output[0].save())
        all_hidden_states.append(layer.output[0].save())

In [7]:
all_hidden_states_numpy = []
all_attention_states_numpy = []


for HS, AS in zip(all_hidden_states, all_attention_states):
    all_hidden_states_numpy.append(HS.value[0].cpu().detach().numpy())
    atts = AS.value[0].cpu().detach().numpy()
    all_attention_states_numpy.append(atts)
    # all_attention_states_numpy.append(atts.reshape(atts.shape[0], n_heads, -1))
all_hidden_states_numpy = np.array(all_hidden_states_numpy)
all_attention_states_numpy = np.array(all_attention_states_numpy)

In [11]:
import numpy as np

og_layer = np.load('/home/wxy/project/honest_llama/features/gsm8k/llama2_chat_7B_gsm8k_head_wise.npy')

In [16]:
og_layer[0]

array([[ 5.3883e-05, -3.8052e-03,  3.1891e-03, ...,  6.6376e-04,
        -1.6441e-03,  1.1282e-03],
       [-3.8177e-02,  1.9623e-02,  3.6377e-02, ..., -2.9316e-03,
        -6.0844e-04, -1.2484e-03],
       [ 2.5806e-03, -7.9727e-04,  4.2648e-03, ...,  2.4109e-03,
         3.2597e-03, -8.3327e-05],
       ...,
       [-2.1801e-03, -1.6270e-03, -1.2306e-02, ...,  9.3994e-03,
         5.9235e-02,  2.2411e-03],
       [-1.2505e-02, -1.0963e-02, -2.6566e-02, ...,  3.8989e-01,
         2.5269e-01, -3.5919e-02],
       [ 6.1768e-02,  4.1260e-02, -1.3525e-01, ...,  2.7618e-02,
        -1.1169e-01, -7.3547e-02]], dtype=float16)

In [17]:
all_attention_states_numpy[:, -1, :]

array([[ 2.4891e-03, -6.3944e-04,  3.9825e-03, ...,  7.7171e-03,
        -5.9738e-03, -2.9736e-03],
       [-3.8624e-03,  1.2962e-02,  1.1396e-03, ...,  3.1877e-04,
         5.8975e-03, -1.9760e-03],
       [ 9.1324e-03,  1.4854e-02,  8.4102e-05, ..., -3.4332e-03,
         3.5400e-03,  8.2779e-03],
       ...,
       [ 1.5587e-02, -4.3762e-02, -1.3000e-01, ..., -1.0065e-01,
        -1.7624e-02, -1.3074e-01],
       [-4.0009e-02,  1.2622e-01,  2.6416e-01, ..., -2.5806e-01,
         1.7822e-01, -2.8641e-02],
       [ 7.8857e-02,  5.1041e-03,  3.8605e-02, ..., -1.8872e-01,
        -2.4280e-01,  1.5479e-01]], dtype=float16)